<a href="https://colab.research.google.com/github/AyushKhatri-Dev/flyrank-search-intelligence/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyushKhatri-Dev/flyrank-search-intelligence/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Which visible pages are showing up in search but not getting as many clicks as
they should?

**The decision this supports.** A content team can't review every page. Each week
they can open a limited number, say fifty. The decision is which fifty.

**Who acts and what they do.** A content or SEO reviewer opens each page from the
list and checks the title and meta description against what the page actually
offers, then rewrites them if there's a clear mismatch.

**Why not just sort by CTR.** A page at position 2 gets a higher CTR than a page
at position 12 anyway, because of where it appears on the results page. So
sorting by raw CTR gives a list of pages ranked low in search, not a list of
pages that underperform. I compare each page only against pages at a similar
position, and rank by the gap.

**What a wrong call costs.** A reviewer edits a page that was fine, and a page
that needed attention stays broken another week. Since reviewers only get
through the top of the list, a wrong page near the top is expensive. That's why
I measure precision@50 and not overall accuracy.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank internship warehouse on Hugging Face, build id
`flyrank_pseudonymized_warehouse_release_v20260703`. Gated, pseudonymized, and
observable signals only — no client names, domains, URLs, or queries anywhere in
it or in this work.

**Tables:** `fact_content_daily_performance` for daily search signals, and
`dim_content` joined on `content_hash_id` for page metadata.

**Windows:** March 2026 for features, April 2026 for evaluation only. A mid-panel
month because the panel is unbalanced and clients start tracking at different
dates. June 2026 is the final month and the natural outcome window for any
past-to-future label, so it stays sealed — which also rules out the `_sample`
table, since that table is exactly June.

**Grain.** The daily table is one row per page per client per day. That's not the
decision grain — a reviewer acts on a page, not a page-day. So I aggregate to one
row per client-page over the month. Verified in ML-04: 9,841,378 daily rows and
9,841,378 distinct date+client+content keys, collapsing to 331,437 client-pages.

**Filters:** at least 100 impressions, average position 1-20, and content created
before the window opened. After filtering: 70,075 pages across 38 clients in
March.

**What I excluded and why:**

- `days_since_update` and `content_updated_date`. 56,874 of 70,075 values fall
  after my window start, and the most common single value is shared by 12,734
  pages. It reads as a bulk sync stamp, not an edit history. Using it would put
  future information into a rule meant to work at the decision moment.
- `fact_content_query_90d`. Its window is a fixed 90 days I don't control, and it
  overlaps any target window I define.
- `sessions_ai` and the AI platform columns. 30,177 rows carry AI sessions across
  a 78.8M-row table. Too sparse to be stable at my grain.
- `gsc_clicks` as a feature. CTR is clicks over impressions and impressions is
  already a feature, so clicks is half my own label. Demonstrated in ML-04:
  adding it moved R² from 0.012 to 0.993.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [4]:
import os, json
import duckdb, pandas as pd, numpy as np
from google.colab import userdata
from huggingface_hub import login
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupShuffleSplit

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
CONTENT = f"read_parquet('{BASE}/dim_content.parquet')"

def build_frame(month, window_start, with_meta=True):
    daily = f"read_parquet('{BASE}/fact_content_daily_performance/month={month}/data_0.parquet')"
    meta = f"""
        , c.content_type, c.main_intent, c.word_count,
        DATE_DIFF('day', c.content_created_date, DATE '{window_start}') AS content_age_days
    """ if with_meta else ""
    join = f"JOIN {CONTENT} c USING (client_hash_id, content_hash_id)" if with_meta else ""
    age_filter = f"AND DATE_DIFF('day', c.content_created_date, DATE '{window_start}') >= 0" if with_meta else ""

    return con.sql(f"""
        WITH agg AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(gsc_impressions) AS impressions,
                   SUM(gsc_clicks) AS clicks,
                   SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions) AS avg_position
            FROM {daily}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT a.*, a.clicks * 100.0 / a.impressions AS ctr {meta}
        FROM agg a {join}
        WHERE a.impressions >= 100
          AND a.avg_position BETWEEN 1 AND 20
          {age_filter}
    """).df()

# Feature month — everything the model sees
march = build_frame("2026-03", "2026-03-01")

# Evaluation month — used ONLY to score the ranking, never as a feature
april = build_frame("2026-04", "2026-04-01", with_meta=False)
march = march.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
april = april.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

for df in (march, april):
    df["position_tier"] = pd.cut(df["avg_position"], [0, 3, 10, 20],
                                 labels=["1-3", "4-10", "11-20"])
    df["tier_median_ctr"] = df.groupby("position_tier", observed=True)["ctr"].transform("median")
    df["below_tier"] = (df["ctr"] < df["tier_median_ctr"]).astype(int)

print(f"March (features):   {len(march):,} pages, {march['client_hash_id'].nunique()} clients")
print(f"April (evaluation): {len(april):,} pages, {april['client_hash_id'].nunique()} clients")

overlap = march.merge(april[["client_hash_id", "content_hash_id", "below_tier"]],
                      on=["client_hash_id", "content_hash_id"], suffixes=("", "_april"))
print(f"Pages present in both months: {len(overlap):,} ({len(overlap)/len(march)*100:.1f}% of March)")
print(f"\nOf March pages that were below their tier median, still below in April: "
      f"{overlap[overlap['below_tier']==1]['below_tier_april'].mean()*100:.1f}%")

# ---------- Evaluation label: observed in April, never a feature ----------
data = march.merge(
    april[["client_hash_id", "content_hash_id", "below_tier"]].rename(
        columns={"below_tier": "y_april"}),
    on=["client_hash_id", "content_hash_id"])

print(f"Evaluable pages (in both months): {len(data):,}")
print(f"Base rate — below tier median in April: {data['y_april'].mean()*100:.1f}%\n")

# ---------- Client-holdout split ----------
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
              .split(data, groups=data["client_hash_id"]))
train, test = data.iloc[tr].copy(), data.iloc[te].copy()
print(f"Train: {len(train):,} pages / {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test):,} pages / {test['client_hash_id'].nunique()} clients\n")

# ---------- Model: predict clicks with Poisson loss, impressions as exposure ----------
FEATS = ["avg_position", "impressions", "content_age_days", "word_count",
         "content_type", "main_intent"]

def prep(df):
    X = pd.get_dummies(df[FEATS], columns=["content_type", "main_intent"], dummy_na=True)
    return X

X_tr, X_te = prep(train), prep(test)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

model = HistGradientBoostingRegressor(loss="poisson", random_state=42)
model.fit(X_tr, train["clicks"])

test["predicted_clicks"] = np.maximum(model.predict(X_te), 0)
test["predicted_ctr"] = test["predicted_clicks"] * 100 / test["impressions"]
test["model_score"] = ((test["predicted_ctr"] - test["ctr"]).clip(lower=0)
                       * test["impressions"] / 100)

# ---------- Baseline: the ML-05 rule, same test set ----------
test["baseline_score"] = ((test["tier_median_ctr"] - test["ctr"]).clip(lower=0)
                          * test["impressions"] / 100)

# ---------- Carry-forward: the cheapest possible ranker ----------
test["carry_score"] = test["below_tier"]

def precision_at_k(scores, y, k):
    return y.iloc[np.argsort(-scores.values)[:k]].mean()

print("Precision@K — of the top K pages, how many were still below their tier median in April\n")
print(f"{'K':>6} {'carry-fwd':>11} {'baseline':>10} {'model':>8}")
for k in (20, 50, 100, 500):
    print(f"{k:>6} "
          f"{precision_at_k(test['carry_score'], test['y_april'], k):>11.3f} "
          f"{precision_at_k(test['baseline_score'], test['y_april'], k):>10.3f} "
          f"{precision_at_k(test['model_score'], test['y_april'], k):>8.3f}")

print(f"\nBase rate on test set: {test['y_april'].mean():.3f}")
print("\nTier mix in top 50:")
print(pd.concat([
    test.iloc[np.argsort(-test['baseline_score'].values)[:50]]["position_tier"].value_counts().rename("baseline"),
    test.iloc[np.argsort(-test['model_score'].values)[:50]]["position_tier"].value_counts().rename("model"),
], axis=1).fillna(0).astype(int).to_string())

from scipy.stats import spearmanr

april_ctr = april.set_index(["client_hash_id", "content_hash_id"])["ctr"]
test["april_ctr"] = test.set_index(["client_hash_id", "content_hash_id"]).index.map(april_ctr)
test["april_clicks"] = test.set_index(["client_hash_id", "content_hash_id"]).index.map(
    april.set_index(["client_hash_id", "content_hash_id"])["clicks"])

def topk(scores, k):
    return test.iloc[np.argsort(-scores.values)[:k]]

print("Neutral check — no tier median in the metric.")
print("Lower April CTR at the top = better at finding genuine underperformers.\n")
print(f"{'K':>6} {'carry-fwd':>11} {'baseline':>10} {'model':>8}")
for k in (20, 50, 100, 500):
    print(f"{k:>6} "
          f"{topk(test['carry_score'], k)['april_ctr'].mean():>11.3f} "
          f"{topk(test['baseline_score'], k)['april_ctr'].mean():>10.3f} "
          f"{topk(test['model_score'], k)['april_ctr'].mean():>8.3f}")

print(f"\nTest-set mean April CTR (the bar to beat): {test['april_ctr'].mean():.3f}")

print("\nShare of top-K with ZERO clicks in April:")
print(f"{'K':>6} {'carry-fwd':>11} {'baseline':>10} {'model':>8}")
for k in (20, 50, 100):
    print(f"{k:>6} "
          f"{(topk(test['carry_score'], k)['april_clicks'] == 0).mean():>11.3f} "
          f"{(topk(test['baseline_score'], k)['april_clicks'] == 0).mean():>10.3f} "
          f"{(topk(test['model_score'], k)['april_clicks'] == 0).mean():>8.3f}")

print("\nSpearman correlation with April CTR (more negative = better ranking):")
for name, col in [("carry-forward", "carry_score"), ("baseline", "baseline_score"), ("model", "model_score")]:
    rho, p = spearmanr(test[col], test["april_ctr"])
    print(f"  {name:15s} rho = {rho:+.3f}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March (features):   70,075 pages, 38 clients
April (evaluation): 77,895 pages, 46 clients
Pages present in both months: 55,925 (79.8% of March)

Of March pages that were below their tier median, still below in April: 74.7%
Evaluable pages (in both months): 55,925
Base rate — below tier median in April: 51.6%

Train: 37,478 pages / 25 clients
Test:  18,447 pages / 12 clients

Precision@K — of the top K pages, how many were still below their tier median in April

     K   carry-fwd   baseline    model
    20       0.900      0.850    0.600
    50       0.740      0.860    0.680
   100       0.720      0.910    0.740
   500       0.734      0.932    0.812

Base rate on test set: 0.563

Tier mix in top 50:
               baseline  model
position_tier                 
4-10                 39     40
1-3                  11     10
11-20                 0      0
Neutral check — no tier median in the metric.
Lower April CTR at the top = better at finding genuine underperformers.

     K   carry

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*


Three rankers, same held-out clients (12 clients, 18,447 pages), scored against
one observed outcome: was the page still below its position tier's median CTR in
April 2026? April data is used only to score the rankings, never as a feature.

**Metric 1 — Precision@K on the April outcome**

| K | Carry-forward | Baseline rule | Model |
|---:|---:|---:|---:|
| 20 | 0.900 | 0.850 | 0.600 |
| 50 | 0.740 | **0.860** | 0.680 |
| 100 | 0.720 | **0.910** | 0.740 |
| 500 | 0.734 | **0.932** | 0.812 |

Base rate on the test set: 0.563.

**Metric 2 — mean April CTR of the top-K (lower is better, no tier median in it)**

| K | Carry-forward | Baseline rule | Model |
|---:|---:|---:|---:|
| 20 | 0.063 | 0.086 | 0.165 |
| 50 | 0.092 | **0.071** | 0.158 |
| 100 | 0.093 | **0.061** | 0.132 |
| 500 | 0.102 | **0.068** | 0.109 |

Test-set mean April CTR: 0.234. Spearman correlation with April CTR:
carry-forward −0.399, baseline −0.361, model −0.276.

### The model did not beat the baseline

It lost on both metrics. I expected the first metric to favour the baseline —
the label asks "below tier median in April?" and the baseline scores "how far
below tier median in March?", which is the same quantity a month apart. So I
built the second metric to remove that advantage: it only asks what CTR the
top-ranked pages actually had in April, and never touches a tier median.

The baseline won there too. So this is not a metric artefact. On this slice, the
transparent rule ranks better than the learned model.

The reason traces back to ML-04. The model's job is to predict expected clicks
from six features, and those features explained almost none of the variation in
CTR (R² 0.012 on a held-out split). A residual is only as good as the prediction
it is measured against, so a weak predictor produces a noisy residual. Adding a
model on top of a bad expectation made the ranking worse, not better.

### Carry-forward is not really a third ranker

It looks strongest at K=20, but its score is binary, so thousands of pages tie at
1 and its "top 20" is an arbitrary slice of that tie group — which is why its
numbers move between runs while the baseline's do not. Its top-20 also contains
65% pages with zero April clicks, against 5% for the baseline. It is finding
low-volume pages that underperform truly but trivially. A page with 200
impressions is not worth a reviewer's hour.

### What all three share

None of them put a single tier 11-20 page in the top 50 — the same structural
bias I identified in ML-05. Both the baseline and the model multiply a shortfall
by impressions, and tier 11-20 is smaller on both terms. The model inherited the
bias rather than fixing it, which tells me the bias lives in the scoring shape,
not in the choice of estimator.

In [5]:
results = {
    "test_clients": int(test["client_hash_id"].nunique()),
    "test_pages": int(len(test)),
    "base_rate_april": round(float(test["y_april"].mean()), 3),
    "precision_at_k": {
        str(k): {
            "carry_forward": round(float(precision_at_k(test["carry_score"], test["y_april"], k)), 3),
            "baseline": round(float(precision_at_k(test["baseline_score"], test["y_april"], k)), 3),
            "model": round(float(precision_at_k(test["model_score"], test["y_april"], k)), 3),
        } for k in (20, 50, 100, 500)
    },
    "mean_april_ctr_at_k": {
        str(k): {
            "carry_forward": round(float(topk(test["carry_score"], k)["april_ctr"].mean()), 3),
            "baseline": round(float(topk(test["baseline_score"], k)["april_ctr"].mean()), 3),
            "model": round(float(topk(test["model_score"], k)["april_ctr"].mean()), 3),
        } for k in (20, 50, 100, 500)
    },
    "verdict": "baseline beats model on both metrics",
}
os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/capstone_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

{
  "test_clients": 12,
  "test_pages": 18447,
  "base_rate_april": 0.563,
  "precision_at_k": {
    "20": {
      "carry_forward": 0.9,
      "baseline": 0.85,
      "model": 0.6
    },
    "50": {
      "carry_forward": 0.74,
      "baseline": 0.86,
      "model": 0.68
    },
    "100": {
      "carry_forward": 0.72,
      "baseline": 0.91,
      "model": 0.74
    },
    "500": {
      "carry_forward": 0.734,
      "baseline": 0.932,
      "model": 0.812
    }
  },
  "mean_april_ctr_at_k": {
    "20": {
      "carry_forward": 0.063,
      "baseline": 0.086,
      "model": 0.165
    },
    "50": {
      "carry_forward": 0.092,
      "baseline": 0.071,
      "model": 0.158
    },
    "100": {
      "carry_forward": 0.093,
      "baseline": 0.061,
      "model": 0.132
    },
    "500": {
      "carry_forward": 0.102,
      "baseline": 0.068,
      "model": 0.109
    }
  },
  "verdict": "baseline beats model on both metrics"
}


## 5. Limitations

*What this work cannot claim.*

**The availability flag tells me nothing extra.** In March, exactly 3,611,061
rows have `gsc_data_available IS TRUE` and exactly 3,611,061 have impressions
above zero. Identical counts. So I can't separate a page that was tracked and
served zero impressions from one that wasn't tracked at all. For the other 63% of
daily rows I don't know why the data is missing.

**Tier 11-20 never appears.** It has 18,149 pages in March and 43% of them got
zero clicks, the worst outcomes in the slice. Neither the baseline nor the model
puts a single one of them in the top 50, because both multiply a shortfall by
impressions and tier 11-20 is smaller on both terms. The pages most likely to be
underperforming are the ones my queue cannot see.

**The model's numbers move slightly between runs.** Precision@50 came out at
0.680 and 0.720 on two runs of identical code with a fixed random seed — about
two pages out of fifty. The baseline's numbers are identical across runs. I'd
report the model's figures as approximate.

**One month, one slice.** March 2026, 38 clients, 70,075 pages after filtering.
Held-out evaluation used 12 clients. Nothing here has been tested at full
warehouse scale or across seasons.

**What this work can never say.** That editing a page causes clicks to change.
I observe impressions, clicks, and position after the fact — never the ranking
system, the results page, or the user. Even a perfect ranking here is
decision-support: pages worth a reviewer's time, in an order. It is not a causal
claim, and it is not a statement about how Google ranks anything.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The playbook ships the **baseline rule**, not the model. That is the honest
outcome of section 4: the rule ranked better on both metrics, and it is also the
one a reviewer can read.

Every row carries a score in estimated missed clicks, one reason code
(`below_tier_ctr`), one action (`review_title_and_meta`), and a confidence label.

Confidence is where the ML-05 top-20 review changed the design. Seven of those
twenty pages had near-zero CTR at very high exposure — a page at position 1.4
with 44,707 impressions and zero clicks is not a title problem. So near-zero-CTR
pages no longer get "high" confidence. They get **`verify_tracking_first`**: look
at the measurement before editing anything.

**How to use the queue:**

1. Work top-down; the ordering carries the information, not the flag.
2. Start with `high` confidence rows — real gap, real volume, plausibly fixable.
3. For `verify_tracking_first`, check the measurement first. A SERP feature
   answering the query, or a tracking gap, will not be fixed by a rewrite.
4. Stop when the week's capacity runs out. Precision@50 was chosen because
   fifty is a plausible weekly load; the metric matches the decision.

**What would make a recommendation wrong:** the query behind a page may simply
not match what the page offers, which no metadata edit fixes. Pages already at
positions 1-3 have a low ceiling even when the rewrite works. And the queue
cannot see tier 11-20 at all — see section 5.

In [6]:
final = test.sort_values("baseline_score", ascending=False).reset_index(drop=True)
final["rank"] = final.index + 1
final["reason_code"] = "below_tier_ctr"
final["action"] = "review_title_and_meta"
final["confidence"] = np.where(
    (final["impressions"] >= 1000) & (final["ctr"] >= 0.01), "high",
    np.where(final["ctr"] < 0.01, "verify_tracking_first", "medium"))

cols = ["rank", "client_hash_id", "content_hash_id", "position_tier", "avg_position",
        "impressions", "clicks", "ctr", "tier_median_ctr", "baseline_score",
        "reason_code", "action", "confidence"]

os.makedirs("work/outputs", exist_ok=True)
final[cols].to_csv("work/outputs/capstone_action_queue.csv", index=False)

print(f"Queue: {len(final):,} pages\n")
print("Confidence mix in top 100:")
print(final.head(100)["confidence"].value_counts().to_string())
print("\nTop 10:")
print(final.head(10)[["rank", "position_tier", "avg_position", "impressions",
                      "ctr", "baseline_score", "confidence"]].round(3).to_string(index=False))

Queue: 18,447 pages

Confidence mix in top 100:
confidence
high                     88
verify_tracking_first    12

Top 10:
 rank position_tier  avg_position  impressions   ctr  baseline_score            confidence
    1          4-10         3.166     143019.0 0.030         218.703                  high
    2          4-10         9.736     107584.0 0.014         181.862                  high
    3          4-10         5.948     132593.0 0.063         159.625                  high
    4          4-10         8.005      82376.0 0.013         139.736                  high
    5          4-10         6.127      73135.0 0.022         117.826                  high
    6          4-10         8.708      63201.0 0.003         113.648 verify_tracking_first
    7           1-3         2.126      60172.0 0.030         109.214                  high
    8          4-10         8.331      58278.0 0.009         101.640 verify_tracking_first
    9          4-10         4.622      65330.0 0.052     

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs/charts", exist_ok=True)
ks = [20, 50, 100, 500]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for name, col in [("Carry-forward", "carry_score"), ("Baseline rule", "baseline_score"), ("Model", "model_score")]:
    axes[0].plot(ks, [precision_at_k(test[col], test["y_april"], k) for k in ks], marker="o", label=name)
axes[0].axhline(test["y_april"].mean(), ls="--", c="grey", label="Base rate")
axes[0].set(xscale="log", xticks=ks, xticklabels=ks, xlabel="K", ylabel="Precision@K",
            title="Precision@K on the April outcome")
axes[0].legend(); axes[0].grid(alpha=.3)

for name, col in [("Carry-forward", "carry_score"), ("Baseline rule", "baseline_score"), ("Model", "model_score")]:
    axes[1].plot(ks, [topk(test[col], k)["april_ctr"].mean() for k in ks], marker="o", label=name)
axes[1].axhline(test["april_ctr"].mean(), ls="--", c="grey", label="Test mean")
axes[1].set(xscale="log", xticks=ks, xticklabels=ks, xlabel="K",
            ylabel="Mean April CTR (%) — lower is better",
            title="Neutral check: what the top-K actually did")
axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig("work/outputs/charts/model_vs_baseline.png", dpi=150)
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

t = march.groupby("position_tier", observed=True).agg(
    median_ctr=("ctr", "median"), zero=("clicks", lambda s: (s == 0).mean() * 100))
axes[0].bar(t.index.astype(str), t["median_ctr"], color="steelblue")
axes[0].set(xlabel="Position tier", ylabel="Median CTR (%)", title="CTR falls with position (n=70,075)")
ax2 = axes[0].twinx()
ax2.plot(t.index.astype(str), t["zero"], marker="o", color="firebrick")
ax2.set_ylabel("% pages with zero clicks", color="firebrick")

mix = pd.concat([
    final.head(50)["position_tier"].value_counts().rename("Baseline"),
    test.iloc[np.argsort(-test["model_score"].values)[:50]]["position_tier"].value_counts().rename("Model"),
], axis=1).fillna(0).reindex(["1-3", "4-10", "11-20"])
mix.plot(kind="bar", ax=axes[1], color=["steelblue", "darkorange"])
axes[1].set(xlabel="Position tier", ylabel="Pages in top 50",
            title="Neither ranker surfaces tier 11-20")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("work/outputs/charts/signal_and_bias.png", dpi=150)
plt.close()

print("Saved:")
print("  work/outputs/charts/model_vs_baseline.png")
print("  work/outputs/charts/signal_and_bias.png")

Saved:
  work/outputs/charts/model_vs_baseline.png
  work/outputs/charts/signal_and_bias.png


## Closing — ML-12: demo, social post, employer summary

### 5-minute demo outline

**0:00–0:45 — The problem.** A content team can review 50 pages a week out of
tens of thousands. Which 50? The obvious answer — sort by CTR, take the worst —
doesn't work, because a page at position 12 has a low CTR no matter how good it
is. Sorting by raw CTR just re-sorts by position.

**0:45–1:30 — The fix, and the data behind it.** Compare each page only against
pages at a similar position. Show the chart: median CTR falls 0.211 → 0.183 →
0.094 across position tiers, and the share of pages with zero clicks climbs from
22% to 43%. That's the premise, measured on 70,075 pages.

**1:30–2:30 — The rule.** Score each page by estimated missed clicks: how far
below its tier median it sits, multiplied by the impressions it actually got.
One reason code, one action, a confidence label. Explain why the score is in
clicks and not in CTR points — an earlier version ranked on the raw gap and put
all 50 top pages in one tier.

**2:30–3:30 — How I checked it.** Rank on March, score on April, hold out whole
clients so the model never sees a client it was trained on. Two metrics: one
using the tier median, one deliberately avoiding it.

**3:30–4:30 — The result, which is not the one I wanted.** The gradient-boosted
model lost to the simple rule on both metrics: precision@50 of 0.68 against 0.86.
Trace it back — the model's features explained almost none of the variation in
CTR, and a residual is only as good as the prediction behind it.

**4:30–5:00 — What I'd fix.** Neither ranker surfaces a single tier 11-20 page,
and that's where 43% of pages get zero clicks. The bias is in the scoring shape,
not the estimator. Scoring relative to the tier median, or ranking within tier,
is the next thing to test.

### Social post cut

> I spent 8 weeks on a search-data project and my model lost to a three-line rule.
>
> The task: rank which pages a content team should review first for weak
> click-through. I built a transparent baseline — compare each page to others at
> a similar search position, score by estimated missed clicks. Then a gradient
> boosting model on top.
>
> Validated on held-out clients, scored against the next month's observed data.
> Baseline precision@50: 0.86. Model: 0.68.
>
> I checked whether my metric was unfair to the model. Built a second one that
> couldn't favour the rule by construction. The rule still won.
>
> The reason was in an earlier notebook: my features explained almost none of the
> variation in click-through rate. A residual is only as good as the prediction
> you measure it against. Layering a model on a weak expectation made the ranking
> noisier.
>
> Shipping the rule. The more useful finding is the one both of them share —
> neither surfaces pages ranked 11-20, which is exactly where 43% of pages get
> zero clicks. That's a flaw in how I score, not in which model I picked.
>
> Built on the FlyRank ML Internship dataset.

### Employer-facing summary (3 sentences)

I built a decision-support ranking system on 70,075 pages of real search data,
scoring which pages a content team should review first for under-captured clicks,
and validated it on held-out clients against the following month's observed
outcomes. The gradient-boosted model lost to my transparent baseline on both
metrics I tested — precision@50 of 0.68 against 0.86 — and I traced the cause to
a weak underlying prediction rather than reporting the number and moving on. I
shipped the rule, documented the structural bias both approaches share, and named
the specific change I'd test next.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
